In [333]:
import pandas as pd
import plotly.express as px
import plotly.io as pio
import plotly.graph_objects as go
import os
from scipy import stats


In [334]:
base_dir = "/sc/projects/sci-herbrich/chair/lora-bp/vincent.eichhorn/nnt/test/reward_corr/"
df = pd.read_csv(os.path.join(base_dir, "arc_easy/0-savings-0.5-concentration-2.0/reward_log.csv"))
# df = df[["min_layer_id", "loss_improvement"]]
df.dropna(inplace=True)
print(df)

    step  min_layer_id  fisher_score
0      0             0      0.000000
1      1            15      1.394653
2      2             8     15.387543
3      3             8     16.248415
4      4            15      1.414675
..   ...           ...           ...
69    69             0      0.315796
70    70            14      0.042249
71    71             7      0.299064
72    72             2      0.662867
73    73            11      0.082253

[74 rows x 3 columns]


In [ ]:
field = "fisher_score"  # Change this to the field you want to analyze
# df[field] /= (17 - df["min_layer_id"])  # Normalize by the number of layers not frozen
means = df.groupby("min_layer_id")[field].mean()
stds = df.groupby("min_layer_id")[field].std()
# fig = go.Figure()
# fig.add_trace(
#     go.Scatter(
#         x=means.index,
#         y=means.values,
#         mode="markers",
#         marker=dict(color="red", size=10),
#         error_y=dict(type="data", array=stds.values, visible=True),
#         name="Mean ± 1 Std"
#     )
# )
# fig.show()

#drop outliers with are not in the range of mean ± 1 std
for i in df["min_layer_id"].unique():
    mean = means[i]
    std = stds[i]
    df = df[(df["min_layer_id"] != i) | ((df[field] >= mean - std) & (df[field] <= mean + std))]
fig = px.scatter(
    df,
    x="min_layer_id",
    y=field,
    labels={"min_layer_id": "Min Layer ID", field: field.replace("_", " ").title()},
)
fig.show()
corr = stats.pearsonr(df["min_layer_id"], df[field])
print(f"Pearson correlation coefficient: {corr[0]}, p-value: {corr[1]}")


Pearson correlation coefficient: -0.19190479157704982, p-value: 0.10142465583632387


In [336]:
from nnt.util.functions import load_json


aggregated_fisher = load_json(os.path.join(base_dir, "arc_easy/0-savings-0.5-concentration-2.0/fisher_information.json"))
print(aggregated_fisher)

FileNotFoundError: [Errno 2] No such file or directory: '/sc/projects/sci-herbrich/chair/lora-bp/vincent.eichhorn/nnt/test/reward_corr/arc_easy/0-savings-0.5-concentration-2.0/fisher_information.json'